# Movement Classification: Stratified 5-Fold Cross-Validation Sandbox

This notebook performs a rigorous **Stratified 5-Fold Cross-Validation** evaluation across all 115 matched transcription files in `transcriptions/`.

Using cross-validation ensures that every document is used for both training and validation, producing highly reliable, stable, and unbiased performance metrics. We compare:
1. **Rule-Based Regex Classifier**
2. **TF-IDF + Naive Bayes**
3. **TF-IDF + Logistic Regression**
4. **TF-IDF + Support Vector Machine (Linear SVM)**
5. **HuggingFace Semantic Embeddings (Sentence Transformers) + Linear SVM**

---

In [1]:
# 1. Imports
import os
import time
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sentence_transformers import SentenceTransformer

### 2. Load Dataset
We load all 115 matched documents from the Parquet dataset generated by the OCR mapping pipeline.

In [2]:
parquet_path = "files_dataset.parquet"
if not os.path.exists(parquet_path):
    raise FileNotFoundError("files_dataset.parquet is missing. Run build_dataset.py first!")
    
df = pd.read_parquet(parquet_path)
print(f"Loaded dataset with {len(df)} aligned documents.")
print("\nMovement Label Distribution:")
print(df["label_movement"].value_counts())

X = df["extracted_text"].values
y = df["label_movement"].values

Loaded dataset with 115 aligned documents.

Movement Label Distribution:
label_movement
ALTA    94
BAJA    21
Name: count, dtype: int64


### 3. Setup Stratified 5-Fold Cross-Validation
We initialize a Stratified 5-Fold split (shuffled with a fixed seed for exact reproducibility).

In [3]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
results_dict = {}

def run_cv(model_func, is_tfidf=False, X_emb=None):
    accs = []
    f1s = []
    times = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        start_time = time.time()
        
        if X_emb is not None:
            # Embedding model
            X_train_fold, X_test_fold = X_emb[train_idx], X_emb[test_idx]
            y_train_fold, y_test_fold = y[train_idx], y[test_idx]
            
            clf = model_func()
            clf.fit(X_train_fold, y_train_fold)
            preds = clf.predict(X_test_fold)
            
        elif is_tfidf:
            # TF-IDF model
            X_train_fold, X_test_fold = X[train_idx], X[test_idx]
            y_train_fold, y_test_fold = y[train_idx], y[test_idx]
            
            vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
            X_train_vec = vectorizer.fit_transform(X_train_fold)
            X_test_vec = vectorizer.transform(X_test_fold)
            
            clf = model_func()
            clf.fit(X_train_vec, y_train_fold)
            preds = clf.predict(X_test_vec)
            
        else:
            # Rule-based model
            X_test_fold = X[test_idx]
            y_test_fold = y[test_idx]
            preds = [model_func(txt) for txt in X_test_fold]
            
        elapsed = time.time() - start_time
        accs.append(accuracy_score(y_test_fold, preds))
        f1s.append(f1_score(y_test_fold, preds, average="macro"))
        times.append(elapsed)
        
    return {
        "acc_mean": np.mean(accs) * 100,
        "acc_std": np.std(accs) * 100,
        "f1_mean": np.mean(f1s),
        "f1_std": np.std(f1s),
        "time_mean": np.mean(times)
    }

### 4. Evaluate Rule-Based Classifier

In [4]:
def rule_based_classify(text):
    txt = text.lower()
    if any(kw in txt for kw in ["elimina provisionalmente", "suspension definitiva", "dejar sin efecto"]):
        return "BAJA"
    return "ALTA"

results_dict["Rule-Based Regex"] = run_cv(rule_based_classify, is_tfidf=False)
print("Rule-Based Regex CV Completed.")

Rule-Based Regex CV Completed.


### 5. Evaluate Traditional ML Models (TF-IDF)

In [5]:
# A. Naive Bayes
results_dict["TF-IDF + Naive Bayes"] = run_cv(lambda: MultinomialNB(), is_tfidf=True)

# B. Logistic Regression
results_dict["TF-IDF + Logistic Regression"] = run_cv(
    lambda: LogisticRegression(class_weight="balanced", random_state=42), 
    is_tfidf=True
)

# C. Linear SVM
results_dict["TF-IDF + Linear SVM"] = run_cv(
    lambda: SVC(kernel="linear", class_weight="balanced", random_state=42), 
    is_tfidf=True
)
print("TF-IDF Classifiers CV Completed.")

TF-IDF Classifiers CV Completed.


### 6. Evaluate Advanced Semantic Model (Sentence Transformers)

In [6]:
print("Loading local Sentence Transformer model...")
model_name = "local_models/paraphrase-multilingual-MiniLM-L12-v2"
if not os.path.exists(model_name):
    model_name = "paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(model_name)

print("Generating dense embeddings for all 115 documents...")
X_emb = embedder.encode(df["extracted_text"].tolist(), show_progress_bar=True)

results_dict["Semantic Embeddings + SVM"] = run_cv(
    lambda: SVC(kernel="linear", class_weight="balanced", random_state=42), 
    X_emb=X_emb
)
print("Semantic Embeddings SVM CV Completed.")

Loading local Sentence Transformer model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generating dense embeddings for all 115 documents...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Semantic Embeddings SVM CV Completed.


### 7. Compare Cross-Validation Performance

In [7]:
cv_results = []
for model_name, metrics in results_dict.items():
    cv_results.append({
        "Model": model_name,
        "Mean Accuracy (%)": f"{metrics['acc_mean']:.2f}% ± {metrics['acc_std']:.2f}%",
        "Mean F1 (Macro)": f"{metrics['f1_mean']:.4f} ± {metrics['f1_std']:.4f}",
        "Avg Fit Time (s)": f"{metrics['time_mean']:.4f}s"
    })
    
df_results = pd.DataFrame(cv_results)
display(df_results)

,Model,Mean Accuracy (%),Mean F1 (Macro),Avg Fit Time (s)
0,Rule-Based Regex,83.48% ± 1.74%,0.5274 ± 0.0931,0.0016s
1,TF-IDF + Naive Bayes,81.74% ± 1.74%,0.4497 ± 0.0053,0.1026s
2,TF-IDF + Logistic Regression,99.13% ± 1.74%,0.9832 ± 0.0337,0.1529s
3,TF-IDF + Linear SVM,100.00% ± 0.00%,1.0000 ± 0.0000,0.1712s
4,Semantic Embeddings + SVM,70.43% ± 14.39%,0.5456 ± 0.1362,0.0087s
